In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/My Drive/volleypred/'
    print("Running in Colab. Drive mounted.")
except ImportError:
    BASE_PATH = '../'
    print("Running locally. Using relative paths.")

DATA_DIR = os.path.join(BASE_PATH, 'data/')
PLOT_DIR = os.path.join(BASE_PATH, 'plots/')

for folder in [DATA_DIR, PLOT_DIR]:
    if not os.path.exists(folder):
        os.makedirs(folder)
        print(f"Created directory: {folder}")

In [ ]:
df = pd.read_csv(DATA_DIR + 'final_df.csv')
df.head()
df.columns

# Deleting the match

In [ ]:
bedzin_21_22 = df[
    (df['Season'] == '2021/2022') &
    ((df['Team_T1'] == 'MKS Będzin') | (df['Team_T2'] == 'MKS Będzin'))
]

df.drop(bedzin_21_22.index, inplace=True)

# Encoding

In [ ]:
results = {'3:0': 0,
           '3:1': 1,
           '3:2': 2,
           '2:3': 3,
           '1:3': 4,
           '0:3': 5}

df_encoded = df.copy()
df_encoded['Result'] = df_encoded['Score'].map(results)
df_encoded.head()

# Feature selection

In [ ]:
df_selected = df_encoded.copy()
df_selected.drop(columns=['Date', 'Team_T1', 'Team_T2', 'Score'], inplace=True)
df_selected.head()
df_selected.columns

In [ ]:
to_drop = ['Diff_Sum', 'Diff_Srv_Sum', 'Diff_Srv_Ace', 'Diff_Rec_Sum',
           'Diff_Att_Sum', 'Diff_Att_Kill', 'Diff_Blk_As']

df_selected.drop(columns=to_drop, inplace=True)
df_selected.head()

# Scaling

In [ ]:
df_scaled = df_selected.copy()

scaler = StandardScaler()
features_to_scale = [col for col in df_scaled.columns if col not in ['Result', 'Season']]
df_scaled[features_to_scale] = scaler.fit_transform(df_scaled[features_to_scale])

df_scaled.head()

# Saving

In [ ]:
df_model = df_scaled.copy()
df_model.to_csv(DATA_DIR + 'df_model.csv', index=False)